In [1]:
import pandas as pd

df = pd.read_csv(r'../../datasets/CA_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,0,2,CA,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.000000,0.2480,0.576,138.008,4,About_Average
1,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,2,0,3,CA,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.000000,0.1410,0.214,101.061,4,Higher
2,3GCdLUSnKSMJhs4Tj6CV3s,All The Stars (with SZA),"Kendrick Lamar, SZA",3,1,19,CA,2025-02-17,90,True,...,-4.946,1,0.0599,0.0612,0.000195,0.0926,0.557,96.782,4,About_Average
3,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",4,2,-3,CA,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.000000,0.1220,0.535,157.969,3,Lower
4,0aB0v4027ukVziUGwVGYpG,tv off (feat. lefty gunplay),"Kendrick Lamar, Lefty Gunplay",5,0,5,CA,2025-02-17,92,True,...,-6.679,0,0.2630,0.0837,0.000000,0.4230,0.548,100.036,4,Higher


In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [3]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RandomForestRegressor(n_estimators=200, min_samples_split=2, min_samples_leaf=2, max_features=0.8, max_depth=30, n_jobs=4)
)
pipeline.fit(X_train, y_train)

# https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_hist_grad_boosting_comparison.html


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(),
                                                  ['average_song']),
                                                 ('minmaxscaler',
                                                  MinMaxScaler(),
                                                  ['key', 'daily_rank',
                                                   'daily_movement',
                                                   'weekly_movement',
                                                   'is_explicit', 'duration_ms',
                                                   'danceability', 'energy',
                                                   'key', 'loudness', 'mode',
                                                   'speechiness',
                                                   'acousticness',
                                                   'instrumentalness',
                                                   'liveness', 'valence',
                                                   'tempo',
                                                   'time_signature'])])),
                ('randomforestregressor',
                 RandomForestRegressor(max_depth=30, max_features=0.8,
                                       min_samples_leaf=2, n_estimators=200,
                                       n_jobs=4))])

In [4]:
param_distributions = {
    'randomforestregressor__n_estimators': [50, 100, 200, 300],
    'randomforestregressor__max_depth': [5, 10, 20, 30, None],
    'randomforestregressor__min_samples_split': [2, 5, 10],
    'randomforestregressor__min_samples_leaf': [1, 2, 4],
    'randomforestregressor__max_features': ['sqrt', 'log2', 0.8]
}

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold

RFRGrid = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_iter=20,
    random_state=42,
    verbose=3
)

RFRGrid.fit(X_train, y_train)
print("Random Forests Regression with Grid Search")
print(RFRGrid.best_params_)
print(RFRGrid.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END randomforestregressor__max_depth=20, randomforestregressor__max_features=sqrt, randomforestregressor__min_samples_leaf=2, randomforestregressor__min_samples_split=2, randomforestregressor__n_estimators=100;, score=-62.495 total time=   0.4s
[CV 2/5] END randomforestregressor__max_depth=20, randomforestregressor__max_features=sqrt, randomforestregressor__min_samples_leaf=2, randomforestregressor__min_samples_split=2, randomforestregressor__n_estimators=100;, score=-73.106 total time=   0.4s
[CV 3/5] END randomforestregressor__max_depth=20, randomforestregressor__max_features=sqrt, randomforestregressor__min_samples_leaf=2, randomforestregressor__min_samples_split=2, randomforestregressor__n_estimators=100;, score=-50.488 total time=   0.4s
[CV 4/5] END randomforestregressor__max_depth=20, randomforestregressor__max_features=sqrt, randomforestregressor__min_samples_leaf=2, randomforestregressor__min_samples_split=

In [5]:


pipeline = make_pipeline(
    preprocessing,
    RandomForestRegressor(n_estimators=100, min_samples_split=2, min_samples_leaf=1, max_features="log2", max_depth=None, n_jobs=4)
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Random Forests Regression")
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))
print("R2: ", r2_score(y_test, y_pred))


Random Forests Regression
MAE:  2.65446486561157
MSE:  58.669195245781964
RMSE:  7.6595819236941365
R2:  0.5753825151231186
